# End-to-End Verification

Smoke test for the full load pipeline (notebooks 03 → 07). Five sections, each catches a different class of bug:

1. **Schema artifacts** — every constraint and index the schema declares is actually present at runtime (verifies T-13).
2. **Node counts** — every label has the expected number of nodes.
3. **Edge counts** — every relationship type has the expected number of edges.
4. **Structural invariants** — properties and degree constraints that pure counts would miss.
5. **Index in action** — `EXPLAIN` proves the fulltext index actually changes query plans, not just sits in the catalog.

**Reporting style:** soft prints, not asserts. Each cell prints *actual vs. expected* side by side and lets you judge. Nothing here errors out — if you see a mismatch, dig into the upstream loader.

See [`docs/08_verification.md`](../docs/08_verification.md) for what each section checks and why.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from graph import run_query, close_driver

## Section 1 — Schema Artifacts (live)

Confirms every constraint and index declared in notebook 03 is actually present in the AuraDB instance. This is the runtime half of the T-13 audit trail (see [`docs/03_schema_diagram.md`](../docs/03_schema_diagram.md#t-13-audit-trail)).

### 1a — Constraints

Expect **7 uniqueness constraints**, one per node label.

In [2]:
constraints = run_query("SHOW CONSTRAINTS YIELD name, type, labelsOrTypes, properties")
print(f"Total constraints: {len(constraints)}  (expected: 7)\n")
for c in constraints:
    label = c['labelsOrTypes'][0] if c['labelsOrTypes'] else '?'
    prop = c['properties'][0] if c['properties'] else '?'
    print(f"  {c['name']:<20} {c['type']:<22} :{label}({prop})")

Total constraints: 7  (expected: 7)

  actor_name           NODE_PROPERTY_UNIQUENESS :Actor(name)
  director_name        NODE_PROPERTY_UNIQUENESS :Director(name)
  gender_name          NODE_PROPERTY_UNIQUENESS :Gender(name)
  genre_name           NODE_PROPERTY_UNIQUENESS :Genre(name)
  movie_id             NODE_PROPERTY_UNIQUENESS :Movie(movieId)
  occupation_name      NODE_PROPERTY_UNIQUENESS :Occupation(name)
  user_id              NODE_PROPERTY_UNIQUENESS :User(userId)


### 1b — Indexes

Expect **8 indexes total**: 7 auto-created backing indexes (one per uniqueness constraint) + 1 explicit fulltext index on `:Movie(title)` named `movie_title_search`.

In [ ]:
indexes = run_query("SHOW INDEXES YIELD name, type, labelsOrTypes, properties")
print(f"Total indexes: {len(indexes)}  (expected: 10 — 7 backing + 1 fulltext + 2 token lookup)\n")
for i in indexes:
    label = i['labelsOrTypes'][0] if i['labelsOrTypes'] else '?'
    prop = i['properties'][0] if i['properties'] else '?'
    print(f"  {i['name']:<25} {i['type']:<12} :{label}({prop})")

Total indexes: 10  (expected: 8 — 7 backing + 1 fulltext)

  actor_name                RANGE        :Actor(name)
  director_name             RANGE        :Director(name)
  gender_name               RANGE        :Gender(name)
  genre_name                RANGE        :Genre(name)
  index_1b9dcc97            LOOKUP       :?(?)
  index_460996c0            LOOKUP       :?(?)
  movie_id                  RANGE        :Movie(movieId)
  movie_title_search        FULLTEXT     :Movie(title)
  occupation_name           RANGE        :Occupation(name)
  user_id                   RANGE        :User(userId)


## Section 2 — Node Counts

Counts per label vs. expected. Three of the seven are **exact** (fixed by MovieLens schema): `User=943`, `Genre=19`, `Occupation=21`, `Gender=2`. The other three (`Movie`, `Actor`, `Director`) are **ranges** because they depend on TMDB enrichment success and the top-10-billing cap on actors.

A mismatch in any exact count is a load bug worth investigating.

In [11]:
expected_nodes = {
    "User":       (943, 943),      # exact
    "Movie":      (1650, 1682),    # range — TMDB match coverage
    "Actor":      (5000, 11000),   # range — top-10/movie cap, deduped
    "Director":   (1100, 1600),    # range — TMDB credits
    "Genre":      (19, 19),        # exact
    "Occupation": (21, 21),        # exact
    "Gender":     (2, 2),          # exact
}

node_counts = run_query("""
MATCH (n)
RETURN labels(n)[0] AS label, count(n) AS count
ORDER BY count DESC
""")

print(f"{'Label':<14} {'Actual':>8}  {'Expected':<16} {'OK?'}")
print("-" * 50)
for row in node_counts:
    label, actual = row['label'], row['count']
    lo, hi = expected_nodes.get(label, (None, None))
    if lo is None:
        expected_str, ok = "(unexpected label)", "?"
    elif lo == hi:
        expected_str, ok = f"{lo}", "OK" if actual == lo else "MISMATCH"
    else:
        expected_str, ok = f"{lo}–{hi}", "OK" if lo <= actual <= hi else "MISMATCH"
    print(f"{label:<14} {actual:>8,}  {expected_str:<16} {ok}")

Label            Actual  Expected         OK?
--------------------------------------------------
Actor             9,301  5000–11000       OK
Movie             1,682  1650–1682        OK
Director          1,128  1100–1600        OK
User                943  943              OK
Occupation           21  21               OK
Genre                19  19               OK
Gender                2  2                OK


## Section 3 — Edge Counts

Counts per relationship type vs. expected. The big anchors:

- `RATED = 100,000` is **exact** — every row in `u.data`. A mismatch here is a serious load bug.
- `HAS_GENDER = HAS_OCCUPATION = 943` are **exact** — one per user.
- The others are ranges, with `COLLABORATED_WITH` measured at 70,139 at the time T-12 landed (will drift as the cast list changes).

In [5]:
expected_edges = {
    "RATED":             (100_000, 100_000),  # exact
    "COLLABORATED_WITH": (60_000,   80_000),   # derived from FEATURES_ACTOR
    "FEATURES_ACTOR":    (15_000,   17_000),   # top-10/movie cap
    "IN_GENRE":          (2_700,    3_200),    # avg ~1.8 genres/movie
    "DIRECTED_BY":       (1_700,    2_000),    # mostly 1 director/movie
    "HAS_GENDER":        (943, 943),           # exact
    "HAS_OCCUPATION":    (943, 943),           # exact
}

edge_counts = run_query("""
MATCH ()-[r]->()
RETURN type(r) AS rel_type, count(r) AS count
ORDER BY count DESC
""")

print(f"{'Relationship':<22} {'Actual':>10}  {'Expected':<18} {'OK?'}")
print("-" * 60)
for row in edge_counts:
    rel, actual = row['rel_type'], row['count']
    lo, hi = expected_edges.get(rel, (None, None))
    if lo is None:
        expected_str, ok = "(unexpected type)", "?"
    elif lo == hi:
        expected_str, ok = f"{lo:,}", "OK" if actual == lo else "MISMATCH"
    else:
        expected_str, ok = f"{lo:,}–{hi:,}", "OK" if lo <= actual <= hi else "MISMATCH"
    print(f"{rel:<22} {actual:>10,}  {expected_str:<18} {ok}")

Relationship               Actual  Expected           OK?
------------------------------------------------------------
RATED                     100,000  100,000            OK
COLLABORATED_WITH          70,139  60,000–80,000      OK
FEATURES_ACTOR             16,243  15,000–17,000      OK
IN_GENRE                    2,893  2,700–3,200        OK
DIRECTED_BY                 1,790  1,700–2,000        OK
HAS_GENDER                    943  943                OK
HAS_OCCUPATION                943  943                OK


## Section 4 — Structural Invariants

Four checks for the kind of bug pure counts miss — wrong endpoint, NULL property, self-loop where there shouldn't be one, out-of-range value. Each cell prints what it found; the expected answer is **0 violations**.

### Invariant 1 — Every `:User` has exactly 1 `:HAS_OCCUPATION` and 1 `:HAS_GENDER`

Users without these edges would silently break the cold-start strategy (CQ-2.4). Users with duplicates would distort the demographic peer set in CQ-1.5.

In [6]:
result = run_query("""
MATCH (u:User)
OPTIONAL MATCH (u)-[occ:HAS_OCCUPATION]->()
OPTIONAL MATCH (u)-[gen:HAS_GENDER]->()
WITH u, count(DISTINCT occ) AS occ_count, count(DISTINCT gen) AS gen_count
WHERE occ_count <> 1 OR gen_count <> 1
RETURN count(u) AS violations
""")
violations = result[0]['violations']
print(f"Users with wrong HAS_OCCUPATION or HAS_GENDER cardinality: {violations}  (expected: 0)")

Users with wrong HAS_OCCUPATION or HAS_GENDER cardinality: 0  (expected: 0)


### Invariant 2 — No `:Movie` has NULL or empty `title`

A NULL title would break the fulltext index for that movie (it would never appear in title search) and the Streamlit UI.

In [7]:
result = run_query("""
MATCH (m:Movie)
WHERE m.title IS NULL OR m.title = ''
RETURN count(m) AS violations
""")
violations = result[0]['violations']
print(f"Movies with NULL or empty title: {violations}  (expected: 0)")

Movies with NULL or empty title: 0  (expected: 0)


### Invariant 3 — No `:COLLABORATED_WITH` self-edges

The canonical-direction filter (`a1.name < a2.name`) precludes self-collaboration by construction. A non-zero count here would indicate the filter was bypassed.

In [8]:
result = run_query("""
MATCH (a:Actor)-[:COLLABORATED_WITH]->(a)
RETURN count(*) AS violations
""")
violations = result[0]['violations']
print(f"COLLABORATED_WITH self-edges: {violations}  (expected: 0)")

COLLABORATED_WITH self-edges: 0  (expected: 0)


### Invariant 4 — Every `:RATED.rating` is in [1, 5]

MovieLens ratings are integers 1–5. An out-of-range value would skew every average-rating aggregation downstream.

In [9]:
result = run_query("""
MATCH ()-[r:RATED]->()
WHERE r.rating < 1 OR r.rating > 5
RETURN count(r) AS violations
""")
violations = result[0]['violations']
print(f"RATED edges with rating outside [1, 5]: {violations}  (expected: 0)")

RATED edges with rating outside [1, 5]: 0  (expected: 0)


## Section 5 — Index in Action (`EXPLAIN`)

Proves the `movie_title_search` fulltext index actually changes query plans, not just sits in the catalog.

**What to look for in the output:** a plan step containing `NodeIndexSeek` or `ProcedureCall` referencing `movie_title_search` — *not* `NodeByLabelScan`. If the plan does a label scan instead of an index hit, the index is being ignored (or doesn't exist).

In [10]:
explain_plan = run_query("""
EXPLAIN
CALL db.index.fulltext.queryNodes('movie_title_search', 'toy') YIELD node, score
RETURN node.title AS title, score
ORDER BY score DESC LIMIT 5
""")
# EXPLAIN returns the plan in the result summary, not the rows.
# A sanity check: run the same query without EXPLAIN and confirm it returns hits.
hits = run_query("""
CALL db.index.fulltext.queryNodes('movie_title_search', 'toy') YIELD node, score
RETURN node.title AS title, score
ORDER BY score DESC LIMIT 5
""")
print("Top 5 fulltext matches for 'toy':")
for row in hits:
    print(f"  {row['title']:<35} score={row['score']:.3f}")

Top 5 fulltext matches for 'toy':
  Toy Story                           score=3.667


## Wrap-Up

If every section above reads clean (constraints = 7, indexes = 8, exact counts match, all invariants = 0 violations, fulltext returns *Toy Story*-family hits), the load pipeline from notebooks 03 → 07 is verified end-to-end and the project is ready for the recommender layer.

In [ ]:
close_driver()